# How AI Agents Think and Act

## Course Plan:

- Lecture 1 — From LLM to Agent: The Simplest Possible Loop — DONE
- Lecture 2 — Memory and RAG — DONE
- **Lecture 3 — Graphs and Planning** ← you are here
- Lecture 4 — Multi-Agent Systems

---

Prerequisites:
- `.env` file with `OPENAI_API_KEY` and `TAVILY_API_KEY`
- `pip install langgraph tavily-python openai python-dotenv`

# Lecture 3 — Structure and Control: Graphs, Chains, and Planning

**Central question:** How do you build something more complex than a single loop? How do you give an agent a *structured process* to follow?

Lectures 1 and 2 built an agent that calls tools and remembers context. But it still had one shape: a loop. Today we ask:
> *What if different situations require different paths?*

The answer is a **graph** — and the data flowing between its nodes is, as always, strings.

---
# Setup

In [ ]:
# Standard imports and path setup
import os, sys, json
from typing import TypedDict, Literal

sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv("../.env")

from openai import OpenAI
from tavily import TavilyClient

# LangGraph core
from langgraph.graph import StateGraph, END

# Display helpers
from IPython.display import Image, display

openai_client: OpenAI = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
tavily_client: TavilyClient = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

print("Setup complete.")

<details>
<summary>Details: <strong>What is LangGraph?</strong></summary>

LangGraph is a library for building **stateful, multi-step** workflows as directed graphs. Each node is a Python function. Edges define the allowed transitions. A special **state dict** is passed through every node — nodes read from it and write back to it.

It sits on top of LangChain but you do not need LangChain to use it. In this notebook we use it directly with raw OpenAI and Tavily calls — no LangChain abstractions.

The key idea: **nodes are computation, edges are control flow, state is communication.**
</details>

---
# Part 1 — A Graph Is Just Nodes and Edges

Before adding any LLM calls, let's build the smallest possible LangGraph and look at what it is.

In [ ]:
# Minimal state: just a single string field
class MinimalState(TypedDict):
    message: str

# Two nodes — each receives state, returns a partial update
def node_a(state: MinimalState) -> dict:
    print(f"  node_a received: '{state['message']}'")
    return {"message": state["message"] + " → A"}

def node_b(state: MinimalState) -> dict:
    print(f"  node_b received: '{state['message']}'")
    return {"message": state["message"] + " → B"}

# Build the graph
builder: StateGraph = StateGraph(MinimalState)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)

# Wire it: start → A → B → end
builder.set_entry_point("node_a")
builder.add_edge("node_a", "node_b")
builder.add_edge("node_b", END)

graph = builder.compile()

# Run it
result: MinimalState = graph.invoke({"message": "hello"})
print(f"\nFinal state: '{result['message']}'")

<details>
<summary>Details: <strong>What does a node actually return?</strong></summary>

A node function receives the **full current state** and returns a **partial update** — a dict with only the keys it wants to change. LangGraph merges this into the existing state before passing it to the next node.

This is important: nodes do not need to echo back every field. They only declare what changed. This makes nodes composable and independent.

A node that returns `{}` is valid — it reads state and produces no output. Useful for logging, side effects, or checkpoints.
</details>

Let's render the graph structure as a diagram.

In [ ]:
# Render the graph topology as a PNG
display(Image(graph.get_graph().draw_mermaid_png()))

<details>
<summary>Details: <strong>Graphs as diagrams</strong></summary>

LangGraph can render its own topology using Mermaid, a text-to-diagram format. This is useful in two ways:

1. **During development**: you can visually inspect that the wiring matches your intention before running anything.
2. **During teaching**: it makes the structure literal and inspectable — no need to read code to understand the flow.

The diagram is derived from the graph definition, not from a trace. It shows all *possible* paths, not the path actually taken on a given run.
</details>

---
# Part 2 — This Is a Chain, Not a Graph

Look at what we just built: A → B → end. There are no choices. Every run takes the same path. This is a **chain** — a sequence of steps with a fixed order.

A chain is fine for simple pipelines. But it cannot handle: *"if the result isn't good enough, try again"* or *"take a different path depending on what happened"*.

**A graph becomes a graph when it has branching.** The mechanism for this in LangGraph is the **conditional edge**.

In [ ]:
# State carries a counter so we can observe the retry loop
class BranchingState(TypedDict):
    value: int
    attempts: int

def process(state: BranchingState) -> dict:
    # Simulate a step that sometimes needs retrying
    attempts: int = state["attempts"] + 1
    value: int = state["value"] + 10
    print(f"  process: attempt {attempts}, value now {value}")
    return {"value": value, "attempts": attempts}

def check(state: BranchingState) -> Literal["retry", "done"]:
    # Routing function: returns the name of the next node
    if state["value"] < 30:
        print(f"  check: value {state['value']} is too low — retry")
        return "retry"
    else:
        print(f"  check: value {state['value']} is good — done")
        return "done"

# Build the graph with a conditional edge
builder2: StateGraph = StateGraph(BranchingState)
builder2.add_node("process", process)
builder2.set_entry_point("process")

# Conditional edge: after process, call check() to decide where to go
builder2.add_conditional_edges(
    "process",
    check,
    {"retry": "process", "done": END}
)

graph2 = builder2.compile()

result2: BranchingState = graph2.invoke({"value": 0, "attempts": 0})
print(f"\nFinal: value={result2['value']}, attempts={result2['attempts']}")

<details>
<summary>Details: <strong>What is a conditional edge?</strong></summary>

`add_conditional_edges(source_node, routing_fn, mapping)` works like this:

1. After `source_node` finishes, LangGraph calls `routing_fn(state)`.
2. `routing_fn` returns a string — the name of a route.
3. `mapping` translates that route name to the actual next node (or `END`).

The routing function is pure Python — it can look at any field in state. It does not call the LLM. It is the place where *your* logic determines control flow, not the model's output.

The distinction matters: the model produces *content*, the graph determines *structure*. Mixing them causes fragile agents.
</details>

In [ ]:
# Render the branching graph — notice the loop back to process
display(Image(graph2.get_graph().draw_mermaid_png()))

---
# Part 3 — The State Dict Is the Language Flow

Now we add an LLM to a node. Notice what changes: the state dict now carries **strings produced by the model** — not just Python values. The interface between nodes is natural language.

In [ ]:
# State schema for the research graph we will build in Parts 4-5
class ResearchState(TypedDict):
    topic: str           # input: what to research
    search_results: str  # output of the search node
    summary: str         # output of the writing node
    attempts: int        # how many search attempts so far

print("State schema defined.")
print("Fields:", list(ResearchState.__annotations__.keys()))

<details>
<summary>Details: <strong>Why TypedDict for state?</strong></summary>

LangGraph accepts any dict-like type for state. `TypedDict` is the idiomatic choice because:

- It is a plain Python dict at runtime — no overhead, no serialization issues.
- Type annotations give you IDE completion and make the schema readable at a glance.
- LangGraph uses the annotations to validate that nodes return keys that exist in the schema.

Look at the fields: `topic`, `search_results`, `summary` — all strings. The model writes to string fields; the graph routes on those strings. Everything flows as text. This is what we mean by *language flow*.
</details>

---
# Part 4 — Building the Research Graph

Three nodes, one conditional branch:

1. **search** — calls Tavily, stores raw results as a string in state
2. **check_results** — routing function: inspects result quality, routes to `write`, back to `search`, or to `give_up`
3. **write** — calls OpenAI to produce a summary from the search results

In [ ]:
# Node 1: search — query Tavily, store results as a string in state
def search_node(state: ResearchState) -> dict:
    topic: str = state["topic"]
    attempts: int = state.get("attempts", 0) + 1
    print(f"[search] attempt {attempts} for: '{topic}'")

    # Call Tavily and get back a list of result dicts
    response: dict = tavily_client.search(query=topic, max_results=5)
    results: list[dict] = response.get("results", [])

    # Flatten to a single string — this is what flows into the next node
    if results:
        combined: str = "\n\n".join(
            f"Source: {r.get('url', 'unknown')}\n{r.get('content', '')}"
            for r in results
        )
    else:
        combined = ""

    return {"search_results": combined, "attempts": attempts}

print("search_node defined.")

<details>
<summary>Details: <strong>Why flatten to a string?</strong></summary>

Tavily returns structured JSON — a list of dicts with `url`, `title`, `content`, etc. We deliberately collapse this into a single string before putting it in state.

Why? Because the next consumer is an LLM, and LLMs consume text. Passing a Python list would require the writing node to serialize it anyway. By flattening early, we make the handoff explicit: the interface between the search node and the writing node is a string.

A side effect: if you inspect state mid-run, you can read what the agent is working with — it's not opaque objects, it's text.
</details>

In [ ]:
# Routing function: is the search result good enough to proceed?
MAX_ATTEMPTS: int = 3
MIN_RESULT_LENGTH: int = 200  # characters

def check_results(state: ResearchState) -> Literal["write", "search", "give_up"]:
    results: str = state.get("search_results", "")
    attempts: int = state.get("attempts", 0)

    # Hard stop: do not loop forever
    if attempts >= MAX_ATTEMPTS:
        print(f"[check] reached max attempts ({MAX_ATTEMPTS}) — giving up")
        return "give_up"

    # Results too thin — retry
    if len(results) < MIN_RESULT_LENGTH:
        print(f"[check] results too thin ({len(results)} chars) — retrying")
        return "search"

    # Good enough — proceed to writing
    print(f"[check] results look good ({len(results)} chars) — writing")
    return "write"

print("check_results defined.")

<details>
<summary>Details: <strong>The MAX_ATTEMPTS guard</strong></summary>

Any loop in a graph needs a termination condition. Without `MAX_ATTEMPTS`, a bad topic or a Tavily outage returning empty results would spin forever.

This is a general principle for agentic systems: **every cycle needs a budget**. The budget can be attempts, tokens, wall-clock time, or cost. Pick one and make it explicit. Silent infinite loops are one of the most common failure modes in production agents.

Notice that the routing function is deterministic Python — it does not call the LLM to decide whether results are good enough. A length heuristic is brittle, but it is *predictable*. Using an LLM as a quality judge is possible and sometimes necessary, but it adds latency and cost to every iteration.
</details>

In [ ]:
# Node 2: write — call OpenAI to summarize the search results
def write_node(state: ResearchState) -> dict:
    topic: str = state["topic"]
    results: str = state["search_results"]
    print(f"[write] summarizing results for '{topic}'")

    # Both topic and results flow into the prompt as strings
    prompt: str = (
        f"You are a research assistant. Based on the following search results, "
        f"write a clear and concise summary about: {topic}\n\n"
        f"Search results:\n{results}\n\n"
        f"Summary (3-5 paragraphs):"
    )

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    summary: str = response.choices[0].message.content or ""
    return {"summary": summary}

# Node 3: give_up — insert a placeholder if search consistently failed
def give_up_node(state: ResearchState) -> dict:
    print("[give_up] search failed after max attempts")
    return {"summary": f"Could not find sufficient information about '{state['topic']}' after {state['attempts']} attempts."}

print("write_node and give_up_node defined.")

<details>
<summary>Details: <strong>The prompt is the interface</strong></summary>

The writing node passes two string fields from state directly into a prompt: `topic` and `search_results`. There is no structured parsing, no schema validation, no type enforcement. The LLM receives a string and returns a string.

This is the interface the whole course has been building toward. The search node and the writing node share no direct Python dependency. The only contract between them is: *`search_results` is a string the writing node can read*. That contract is enforced by convention and prompt engineering, not by a type system.

This is what makes agent systems both powerful (flexible) and fragile (no compiler catches a broken contract).
</details>

In [ ]:
# Assemble the full research graph
research_builder: StateGraph = StateGraph(ResearchState)

# Register all nodes
research_builder.add_node("search", search_node)
research_builder.add_node("write", write_node)
research_builder.add_node("give_up", give_up_node)

# Entry point
research_builder.set_entry_point("search")

# Conditional edge after search: check results and route accordingly
research_builder.add_conditional_edges(
    "search",
    check_results,
    {
        "write":   "write",
        "search":  "search",
        "give_up": "give_up",
    }
)

# Terminal edges
research_builder.add_edge("write", END)
research_builder.add_edge("give_up", END)

research_graph = research_builder.compile()

# Render the diagram
display(Image(research_graph.get_graph().draw_mermaid_png()))

<details>
<summary>Details: <strong>Reading the diagram</strong></summary>

The diagram shows three things:

1. **Linear paths** — `search → write → END` and `search → give_up → END`
2. **The loop** — `search → search` (the retry branch)
3. **The decision point** — the conditional edge label on the `search` node

The visual immediately reveals something a code listing hides: there is a cycle. Cycles mean the graph can run for a variable number of steps depending on runtime outcomes. This is the core difference between a chain (fixed steps) and a graph (variable steps).
</details>

---
# Part 5 — Run the Graph

In [ ]:
# Run the research graph on a topic
TOPIC: str = "recent advances in quantum error correction"

initial_state: ResearchState = {
    "topic": TOPIC,
    "search_results": "",
    "summary": "",
    "attempts": 0,
}

final_state: ResearchState = research_graph.invoke(initial_state)
print("\n--- Final summary ---")
print(final_state["summary"])

In [ ]:
# Inspect intermediate state — what did the search actually find?
print(f"Attempts: {final_state['attempts']}")
print(f"Search results length: {len(final_state['search_results'])} characters")
print("\n--- First 800 characters of raw search results ---")
print(final_state["search_results"][:800])

<details>
<summary>Details: <strong>Full observability from state</strong></summary>

After the graph completes, `final_state` is the state dict as it was last modified. Every field written during the run is there: `search_results` (raw Tavily output), `summary` (model-generated text), `attempts` (how many search calls were made).

This is useful for debugging: if the summary is poor, inspect `search_results` to see what the model was working from. Was the context thin? Irrelevant? The state dict gives you full observability into the run without any separate logging infrastructure.

In more complex graphs, you can use LangGraph's streaming API to inspect state after each node — not just at the end.
</details>

---
# Part 6 — The ReAct Pattern

The research graph has a fixed structure: search (possibly a few times), then write. The *graph* decides when to retry — not the model.

But what if the agent should decide *what* to search for, evaluate results, and choose the next action — all in a loop?

This is the **ReAct** pattern: **Re**ason + **Act**, interleaved. Yao et al. 2022. It is the most widely used pattern for agents that need to plan and adapt during execution.

The shape:
```
Thought → Action → Observation → Thought → Action → ...
```

Each of these is a string in the state dict. The model writes thoughts and action requests; the graph executes actions and writes observations back.

In [ ]:
# ReAct state: a history of thought/action/observation steps as a list of strings
class ReActState(TypedDict):
    topic: str
    history: list[str]  # accumulates: THOUGHT, ACTION, OBSERVATION strings
    final_answer: str
    step_count: int

MAX_STEPS: int = 6

print("ReActState defined.")

In [ ]:
# Think node: ask the model what to do next given the current history
def think_node(state: ReActState) -> dict:
    topic: str = state["topic"]
    history: list[str] = state["history"]
    step: int = state["step_count"] + 1
    print(f"[think] step {step}")

    # Build context from accumulated history strings
    history_text: str = "\n".join(history) if history else "(no history yet)"

    # Prompt instructs the model to reason and choose an action
    system_prompt: str = (
        "You are a research agent. At each step output exactly one of:\n"
        "  THOUGHT: <your reasoning about what to do next>\n"
        "  ACTION: search(<query>)   — to search for information\n"
        "  ACTION: answer(<text>)    — when you have enough for a final answer\n"
        "Output only one line. No other text."
    )
    user_prompt: str = (
        f"Topic: {topic}\n\n"
        f"History so far:\n{history_text}\n\n"
        f"What is your next step?"
    )

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,
    )
    output: str = (response.choices[0].message.content or "").strip()
    print(f"  model: {output}")

    return {"history": history + [output], "step_count": step}

print("think_node defined.")

In [ ]:
# Act node: execute the action the model requested
def act_node(state: ReActState) -> dict:
    history: list[str] = state["history"]

    # The last history entry is the model's most recent output
    last_entry: str = history[-1] if history else ""

    if last_entry.startswith("ACTION: search("):
        # Extract query from search(...)
        query: str = last_entry[len("ACTION: search("):-1].strip()
        print(f"[act] searching: '{query}'")
        response: dict = tavily_client.search(query=query, max_results=3)
        results: list[dict] = response.get("results", [])
        observation: str = "OBSERVATION: " + (
            "\n".join(r.get("content", "") for r in results) or "No results found."
        )
        return {"history": history + [observation]}

    elif last_entry.startswith("ACTION: answer("):
        # Extract the final answer text
        answer: str = last_entry[len("ACTION: answer("):-1].strip()
        print("[act] final answer received")
        return {"final_answer": answer}

    else:
        # Model output was a THOUGHT — nothing to execute, pass through
        return {}

print("act_node defined.")

<details>
<summary>Details: <strong>Parsing model output vs. tool-call API</strong></summary>

The `act_node` parses the model's text output to decide what to execute. This is a manual version of tool use.

Lecture 1 used OpenAI's native tool-call API — the model emits JSON in a structured field, and the SDK parses it. Here we do it manually to make the mechanism visible.

Both have the same fundamental issue: the model might not follow the format. The tool-call API is more reliable because the model is fine-tuned to produce valid JSON in that field. Text parsing is fragile — a stray period or extra word breaks the parser.

In production ReAct implementations, always use the tool-call API rather than parsing free text. What you see here is a pedagogical simplification, not a recommendation.
</details>

In [ ]:
# Routing function: continue the loop or stop?
def should_continue(state: ReActState) -> Literal["think", "end"]:
    # Stop if we have a final answer
    if state.get("final_answer"):
        return "end"
    # Stop if we've hit the step budget
    if state["step_count"] >= MAX_STEPS:
        print(f"[route] step budget exhausted ({MAX_STEPS} steps)")
        return "end"
    return "think"

# Assemble the ReAct graph
react_builder: StateGraph = StateGraph(ReActState)

react_builder.add_node("think", think_node)
react_builder.add_node("act", act_node)

react_builder.set_entry_point("think")
react_builder.add_edge("think", "act")

react_builder.add_conditional_edges(
    "act",
    should_continue,
    {"think": "think", "end": END}
)

react_graph = react_builder.compile()
display(Image(react_graph.get_graph().draw_mermaid_png()))

<details>
<summary>Details: <strong>ReAct as a graph shape</strong></summary>

The diagram shows the canonical ReAct structure: `think → act → (continue?) → think → ...`

Compare to the research graph in Part 4:

- **Research graph**: the *graph structure* controls when to retry (Python logic on result length). The model only writes content.
- **ReAct**: the *model* decides what action to take next. The graph only provides the execution harness and the termination guard.

This is a spectrum from rigid (graph controls everything) to autonomous (model controls everything). ReAct sits toward the autonomous end — more flexible, more likely to go off the rails without a hard step budget.
</details>

In [ ]:
# Run the ReAct agent
react_initial: ReActState = {
    "topic": "the mathematical foundations of transformer attention",
    "history": [],
    "final_answer": "",
    "step_count": 0,
}

react_final: ReActState = react_graph.invoke(react_initial)

print("\n--- Full agent trace ---")
for entry in react_final["history"]:
    print(entry[:300])
    print()

print("\n--- Final answer ---")
print(react_final["final_answer"] or "(no answer produced — step budget exhausted)")

---
# Recap

| Concept | Where it appeared |
|---|---|
| Nodes and edges | Part 1 — minimal graph |
| Chain vs. graph | Part 2 — `add_conditional_edges` introduces branching |
| State dict as language flow | Part 3 — all inter-node communication is strings |
| Conditional retry loop | Parts 4–5 — search → check → retry or write |
| ReAct pattern | Part 6 — think → act → observe, model-driven |
| Step budget / termination | Parts 4 and 6 — every loop needs a hard stop |

**The through-line:** In classical software, interfaces between components are enforced by type systems. In agent systems, the interface is a string. The graph gives structure to what would otherwise be an uncontrolled loop — but the strings flowing through it are still the real interface.

<details>
<summary>Details: <strong>When should you use a graph vs. a plain loop?</strong></summary>

Use a graph when:
- Different outcomes require meaningfully different next steps (branching).
- You need retry or fallback logic separate from the main path.
- The workflow has more than 2–3 steps and you want it inspectable and renderable.
- You want to add checkpointing, streaming, or interruption later (LangGraph supports all of these on top of the same graph definition).

Use a plain loop when:
- The structure is truly linear — every run takes the same steps.
- The logic is simple enough that a graph adds ceremony without clarity.

A 2-node graph is just a function call with extra steps. The abstraction pays off at 4+ nodes with branching. Before that, a loop is cleaner.
</details>